In [ ]:
from langgraph.graph import START,END
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from typing import TypedDict,Annotated,Literal
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel , Field
from langchain_core.messages import BaseMessage,HumanMessage,SystemMessage
from langgraph.checkpoint.memory import MemorySaver # Using this for Persistence
load_dotenv()

True

In [70]:
class Chatbotstate(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]


In [ ]:
checkpointer = MemorySaver() # call the function memorysaver() 
graph = StateGraph(Chatbotstate)


In [72]:
llm = ChatGroq(model="openai/gpt-oss-120b")
def chatbot(state:Chatbotstate):
    messages = state['messages']
    res =llm.invoke(messages)
    return {
        'messages':[res.content]
    }

In [73]:
graph.add_node("Chatbot",chatbot)

In [74]:
graph.add_edge(START,"Chatbot")
graph.add_edge("Chatbot",END)


In [ ]:
chatbot = graph.compile(checkpointer=checkpointer) 
# After the function called we have to assign the checkpointer into the checkpointer attribute in graph.compile()

In [ ]:
thread_id = '1'
while True:
    usermessage = input("type here :")
    print("user :",usermessage)
    if usermessage.strip().lower() in ['exit','quit']:
        break
    config = {'configurable':{ # config object 
        'thread_id':thread_id
    } }
     # thread id for better extraction or easy extraction just enter the threadid accordingly it will give u the stored data into the RAM
    
    
    res = chatbot.invoke({'messages':[HumanMessage(content=usermessage)]}, config=config)

    print("AI:",res['messages'][-1].content)

user : hi my name is bhavya
AI: Hi Bhavya! 👋 Nice to meet you. How can I help you today?
user : wht is my name
AI: Your name is Bhavya. 😊
user : nice to meet u
AI: Nice to meet you too, Bhavya! 😊 Let me know if there's anything you'd like to chat about or need help with.
user : quit
